In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import scipy as sp
import sklearn as sk
import xgboost as xgb

In [3]:
data = pd.read_csv('apartments_for_rent_classified_100K.csv', sep=';', encoding='cp1252')

C:\Users\thund\AppData\Local\Temp\ipykernel_50940\2137846813.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('apartments_for_rent_classified_100K.csv', sep=';', encoding='cp1252')


In [4]:
data.head()

,id,category,title,body,amenities,bathrooms,bedrooms,currency,fee,has_photo,...,price_display,price_type,square_feet,address,cityname,state,latitude,longitude,source,time
0,5668640009,housing/rent/apartment,One BR 507 & 509 Esplanade,"This unit is located at 507 & 509 Esplanade, R...",NaN,1.0,1.0,USD,No,Thumbnail,...,"$2,195",Monthly,542,507 509 Esplanade,Redondo Beach,CA,33.8520,-118.3759,RentLingo,1577360355
1,5668639818,housing/rent/apartment,Three BR 146 Lochview Drive,"This unit is located at 146 Lochview Drive, Ne...",NaN,1.5,3.0,USD,No,Thumbnail,...,"$1,250",Monthly,1500,146 Lochview Dr,Newport News,VA,37.0867,-76.4941,RentLingo,1577360340
2,5668639686,housing/rent/apartment,Three BR 3101 Morningside Drive,This unit is located at 3101 Morningside Drive...,NaN,2.0,3.0,USD,No,Thumbnail,...,"$1,395",Monthly,1650,3101 Morningside Dr,Raleigh,NC,35.8230,-78.6438,RentLingo,1577360332
3,5668639659,housing/rent/apartment,Two BR 209 Aegean Way,"This unit is located at 209 Aegean Way, Vacavi...",NaN,1.0,2.0,USD,No,Thumbnail,...,"$1,600",Monthly,820,209 Aegean Way,Vacaville,CA,38.3622,-121.9712,RentLingo,1577360330
4,5668639374,housing/rent/apartment,One BR 4805 Marquette NE,"This unit is located at 4805 Marquette NE, Alb...",NaN,1.0,1.0,USD,No,Thumbnail,...,$975,Monthly,624,4805 Marquette NE,Albuquerque,NM,35.1038,-106.6110,RentLingo,1577360308


In [5]:
data.columns

Index(['id', 'category', 'title', 'body', 'amenities', 'bathrooms', 'bedrooms',
       'currency', 'fee', 'has_photo', 'pets_allowed', 'price',
       'price_display', 'price_type', 'square_feet', 'address', 'cityname',
       'state', 'latitude', 'longitude', 'source', 'time'],
      dtype='object')

In [6]:
# Now, look only at rent in atlanta
data = data[data['cityname'] == 'Atlanta']
data = data[data['price_type'] == 'Monthly']
data = data.copy()

In [7]:
# Drop features that are not inerently meaninful
data = data.drop('time', axis=1)
# data = data.drop('latitude', axis=1)
# data = data.drop('longitude', axis=1)
data = data.drop('id', axis=1)
data = data.drop('currency', axis=1)
data = data.drop('price_display', axis=1)
data = data.drop('body', axis=1)
data = data.drop('cityname', axis=1)
data = data.drop('state', axis=1)
data = data.drop('category', axis=1)
data = data.drop('price_type', axis=1)
data = data.drop('address', axis=1)
data = data.drop('title', axis=1)

In [8]:
data.columns

Index(['amenities', 'bathrooms', 'bedrooms', 'fee', 'has_photo',
       'pets_allowed', 'price', 'square_feet', 'latitude', 'longitude',
       'source'],
      dtype='object')

In [9]:
data['amenities'].head()

130     Dishwasher,Elevator,Parking,Patio/Deck,Pool,Re...
645                                                  Pool
776     Cable or Satellite,Clubhouse,Dishwasher,Garbag...
988     Cable or Satellite,Dishwasher,Garbage Disposal...
1047    Cable or Satellite,Clubhouse,Dishwasher,Garbag...
Name: amenities, dtype: object

In [10]:
atl_data = data.copy()
atl_data['pets_allowed'] = atl_data['pets_allowed'].fillna('No')
atl_data = atl_data.dropna()

In [75]:
# We need to deal with amenities by one-hot encoding them
# Convert string data in amenities to a list of strings
def parse(column):
    return [column.strip() for item in column.split(',')]
    
# atl_data['amenity_list'] = atl_data['amenities'].map(parse)
# atl_data = atl_data.drop('amenities', axis=1)


In [89]:
# Split into test and training data
X = atl_data.drop('price', axis=1)
y = atl_data['price']


X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2)

categorical = ['amenities', 'fee', 'has_photo','pets_allowed', 'source'] # if undoing one hot, be sure to include 'amenities'!
#numerical_cols = ['bathrooms', 'bedrooms', 'fee', 'has_photo',
       #'pets_allowed', 'square_feet', 'latitude', 'longitude', 'source']

# Now we can target encode!
encode = sk.preprocessing.TargetEncoder(cv=2)

for item in categorical:
    X_train[item] = encode.fit_transform(X_train[[item]], y_train)
    X_test[item] = encode.transform(X_test[[item]])


# Rest indicies otherwise it will NOT work :(
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Rescale numerical values to help with LASSO and NN convergence
scaler = sk.preprocessing.StandardScaler()
scaler.fit(X_train)
X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

"""
# One Hot Encode the values in the amenity column!
mlb = sk.preprocessing.MultiLabelBinarizer(sparse_output=False)
mlb.fit(X_train['amenity_list'])

X_train_OH = mlb.transform(X_train['amenity_list'])
X_test_OH  = mlb.transform(X_test['amenity_list'])
"""

C:\Users\thund\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(
C:\Users\thund\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(
C:\Users\thund\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(
C:\Users\thund\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(
C:\Users\thund\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=2.
  warnings.warn(


"\n# One Hot Encode the values in the amenity column!\nmlb = sk.preprocessing.MultiLabelBinarizer(sparse_output=False)\nmlb.fit(X_train['amenity_list'])\n\nX_train_OH = mlb.transform(X_train['amenity_list'])\nX_test_OH  = mlb.transform(X_test['amenity_list'])\n"

In [93]:
"""
# Create CSVs
train_data = X_train.copy()
train_data['target'] = y_train

test_data = X_test.copy()
test_data['target'] = y_test

train_data.to_csv('training_target_encoded.csv', index=False)
test_data.to_csv('testing_target_encoded.csv', index=False)
"""

In [90]:
X_train.head()
#y_train.shape

,amenities,bathrooms,bedrooms,fee,has_photo,pets_allowed,square_feet,latitude,longitude,source
0,-0.457650,-0.457650,-0.212753,-0.616307,-0.457650,0.89990,1.184478,1.630748,1.630748,-0.457650
1,-0.457650,-0.457650,1.037792,-0.616307,-0.457650,-1.10229,0.443302,-0.613215,-0.613215,-0.457650
2,-0.457650,-0.457650,-0.197154,1.622568,-0.457650,0.91440,0.810521,-0.613215,-0.613215,-0.457650
3,-0.457650,-0.457650,-0.212753,-0.616307,-0.457650,0.89990,-1.015466,1.630748,1.630748,-0.457650
4,2.185078,2.185078,1.077041,-0.616307,2.185078,-1.10229,-0.345039,-0.613215,-0.613215,2.185078


In [92]:
# Implement LASSO Regression and Cross Validate

lasso = sk.linear_model.Lasso()
hyperparams = np.linspace(1,10,10)

lasso_crossval = sk.model_selection.GridSearchCV(lasso, param_grid={'alpha': hyperparams}, cv=5, scoring='neg_mean_squared_error')
lasso_crossval.fit(X_train, y_train)

print(f"Optimal LASSO Hyperparameter: {lasso_crossval.best_params_['alpha']}")
print(f"Best CV MSE Value {-lasso_crossval.best_score_}")

lasso_best = sk.linear_model.Lasso(alpha=lasso_crossval.best_params_['alpha'])
lasso_best.fit(X_train, y_train)

lasso_prediction = lasso_best.predict(X_test)
lasso_MSE = sk.metrics.mean_squared_error(y_test, lasso_prediction)

print(f'Inference MSE is {lasso_MSE}')
print(f'Inference RMSE is {np.sqrt(lasso_MSE)}')

Optimal LASSO Hyperparameter: 1.018018018018018
Best CV MSE Value 190954.21120999014
Inference MSE is 199920.2423542986
Inference RMSE is 447.1244148492661


In [ ]:
"""
Andy's Note: 
I am including inference values for reference only. 
Do NOT use those to assess the best model. We can only use CV MSE to select our model/optimize hyperparameters.
Otherwise, we are creating data leakage :p

Now that we have our best model, lets get to work on 
"""